<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/NDVI_Succ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap
import xarray as xr

In [ ]:
!pip install xee
import xee

In [ ]:
ee.Authenticate()
ee.Initialize(project = 'ee-grmntfrancis0',
              opt_url = 'https://earthengine-highvolume.googleapis.com')

In [ ]:
m = geemap.Map()
m.add("basemap_selector")
m

In [ ]:
geometry = m._draw_control.last_geometry
geometry


In [ ]:
ndvi = ee.ImageCollection("NOAA/CDR/VIIRS/NDVI/V1").select('NDVI','QA').filterDate('2021','2022')
ndvi


In [ ]:
def cloud_mask(img):
  ndvi_img = img.select('NDVI').multiply(0.0001)
  qa_img = img.select('QA')
  cloud = qa_img.bitwiseAnd(1 << 1).neq(0)
  shadow = qa_img.bitwiseAnd(1 << 2).neq(0)
  mask = cloud.Or(shadow).Not()
  return ndvi_img.updateMask(mask).copyProperties(img, img.propertyNames())


In [ ]:
ndvi_mask = ndvi.map(cloud_mask)
ndvi_mask


In [ ]:
ds = xr.open_dataset(
    ndvi_mask,
    engine='ee',
    scale=30,
    crs='EPSG:4326',
    geometry=ee.Geometry.Rectangle([-7.5, -30, 7.5, 30]),
)

In [ ]:
monthly = ds.resample(time = 'M').mean('time')

monthly


In [ ]:
monthly.NDVI.plot(x = 'lon', y = 'lat', col = 'time', col_wrap =4, robust =True)

In [ ]:
!pip install netCDF4
import netCDF4


In [ ]:
ds.to_netcdf('ndvi_daily.nc')
ds


In [ ]:
!pip install dea_tools


In [ ]:
import dea_tools.temporal


In [ ]:
!pip install netCDF4
import netCDF4

In [ ]:
import numpy as np

In [ ]:
import pandas as pd

In [ ]:
import xarray as xr

In [ ]:
phenology = dea_tools.temporal.xr_phenology(
    ds.NDVI,
    stats=['SOS', 'POS', 'EOS', 'Trough', 'vSOS', 'vPOS', 'vEOS', 'LOS', 'AOS', 'ROG', 'ROS'], method_sos='first', method_eos='last', verbose=True
)

phenology

In [ ]:
phenology.SOS.plot(x = 'lon', y = 'lat', robust = True)

phenology.POS.plot(x = 'lon', y = 'lat', robust = True)

phenology.vPOS.plot(x = 'lon', y = 'lat', robust = True)

phenology.LOS.plot(x = 'lon', y = 'lat', robust = True)

In [ ]:
daily = ds.resample(time = 'D').mean('time')

daily


In [ ]:
daily.NDVI.plot(x = 'lon', y = 'lat', col = 'time', col_wrap =4, robust =True)

In [ ]:
ndvi = ee.ImageCollection("NOAA/CDR/VIIRS/NDVI/V1").select('NDVI','QA').filterDate('2021','2022')
ndvi


In [ ]:
yearly = ds.resample(time = 'Y').mean('time')

yearly


In [ ]:
yearly.NDVI.plot(x = 'lon', y = 'lat', col = 'time', col_wrap =1, robust =True)

In [ ]:
ndvi = ee.ImageCollection("NOAA/CDR/VIIRS/NDVI/V1").select('NDVI','QA').filterDate('2023','2024')
ndvi


In [ ]:
monthly = ds.resample(time = 'M').median('time')

monthly



In [ ]:
monthly.NDVI.plot(x = 'lon', y = 'lat', col = 'time', col_wrap =4, robust =True)

